## Variante Beta del Modelo LSTM

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import pandas as pd
import numpy as np
import torch.optim as optim
import time
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

C:\Users\irvin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.2)
  from scipy.sparse import csr_matrix, issparse


In [2]:
torch.manual_seed(23)
np.random.seed(23)

df = pd.read_csv("./data/train_clean.csv")
print(df.head())

print("\n CARGADO DE DATOS ")
df = pd.read_csv("./data/train_clean.csv")
print(f"Dimensiones del dataset: {df.shape}")

# Ver distribución de clases
print("\n Distribución de clases:")
print(df["discourse_effectiveness"].value_counts())

   Unnamed: 0  discourse_id      essay_id discourse_type  \
0           0  0013cc385424  007ACE74B050           Lead   
1           1  9704a709b505  007ACE74B050       Position   
2           2  c22adee811b6  007ACE74B050          Claim   
3           3  a10d361e54e4  007ACE74B050       Evidence   
4           4  db3e453ec4e2  007ACE74B050   Counterclaim   

  discourse_effectiveness                                         text_clean  
0                Adequate  hi isaac going writing face mar natural landfo...  
1                Adequate  perspective think face natural landform dont t...  
2                Adequate  think face natural landform no life mar descov...  
3                Adequate  life mar would know reason think natural landf...  
4                Adequate  people thought face formed alieans thought lif...  

 CARGADO DE DATOS 
Dimensiones del dataset: (36765, 6)

 Distribución de clases:
discourse_effectiveness
Adequate       20977
Effective       9326
Ineffective     6

In [3]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['discourse_effectiveness'])
print(f"\nMapping de clases: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


Mapping de clases: {'Adequate': np.int64(0), 'Effective': np.int64(1), 'Ineffective': np.int64(2)}


In [4]:
print("\n PREPROCESAMIENTO ")

# Tokenizador
def tokenizer(text):
    text = str(text).lower()

    text = text.replace('<br />', ' ').replace('\n', ' ').replace('\t', ' ')

    tokens = text.split()
    return tokens

# Aplicar tokenización
df['tokens'] = df['text_clean'].apply(tokenizer)

# Construir vocabulario
def build_vocab(token_lists, min_freq=2, max_vocab_size=35000):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    # Ordenar por frecuencia y limitar tamaño
    most_common = counter.most_common(max_vocab_size-2)

    vocab = {'<pad>': 0, '<unk>': 1}
    for idx, (word, count) in enumerate(most_common):
        if count >= min_freq:
            vocab[word] = idx + 2

    print(f"Tamaño del vocabulario: {len(vocab)}")
    print(f"Palabras más comunes: {list(vocab.keys())[:10]}")
    return vocab

vocab = build_vocab(df['tokens'])
vocab_size = len(vocab)

# Convertir a índices y aplicar padding con longitud dinámica
def tokens_to_indices(tokens_list, vocab, max_length=200):
    sequences = []
    for tokens in tokens_list:
        indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
        # No truncar demasiado agresivamente
        indices = indices[:max_length]
        sequences.append(torch.tensor(indices, dtype=torch.long))

    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=vocab['<pad>'])
    return padded_sequences

# Convertir textos a tensores con longitud más generosa
X = tokens_to_indices(df['tokens'], vocab, max_length=200)
y = torch.tensor(df['label'].values, dtype=torch.long)

print(f"Forma de X: {X.shape}")
print(f"Forma de y: {y.shape}")


 PREPROCESAMIENTO 
Tamaño del vocabulario: 11495
Palabras más comunes: ['<pad>', '<unk>', 'student', 'not', 'would', 'people', 'vote', 'school', 'electoral', 'college']
Forma de X: torch.Size([36765, 200])
Forma de y: torch.Size([36765])


In [5]:
print("\n DIVISIÓN DE DATOS CON BALANCEO ")

# División estratificada
train_idx, temp_idx = train_test_split(
    range(len(X)),
    test_size=0.3,
    stratify=y.numpy(),  # Convertir a numpy para stratify
    random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=y.numpy()[temp_idx],  # Convertir a numpy para stratify
    random_state=42
)

# Crear datasets
train_dataset = TensorDataset(X[train_idx], y[train_idx])
val_dataset = TensorDataset(X[val_idx], y[val_idx])
test_dataset = TensorDataset(X[test_idx], y[test_idx])

print(f"Entrenamiento: {len(train_dataset)} ejemplos")
print(f"Validación: {len(val_dataset)} ejemplos")
print(f"Prueba: {len(test_dataset)} ejemplos")

# Calcular pesos para balancear clases
class_counts = df['discourse_effectiveness'].value_counts().sort_index()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)

print(f"\nPesos de clases para balanceo: {class_weights.to_dict()}")

# Weighted Random Sampler para entrenamiento - CORREGIDO
train_labels = y[train_idx].numpy()  # Convertir a numpy array

# Crear un mapeo de índices numéricos a pesos
class_weights_dict = {
    0: class_weights['Adequate'],    # Adequate -> 0
    1: class_weights['Effective'],   # Effective -> 1
    2: class_weights['Ineffective']  # Ineffective -> 2
}

# Asignar pesos usando el mapeo
sample_weights = [class_weights_dict[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)



 DIVISIÓN DE DATOS CON BALANCEO 
Entrenamiento: 25735 ejemplos
Validación: 5515 ejemplos
Prueba: 5515 ejemplos

Pesos de clases para balanceo: {'Adequate': 0.46185738273337573, 'Effective': 1.0388572075485765, 'Ineffective': 1.4992854097180477}


In [6]:
# %% [markdown]
# CELDA ÚNICA: Modelo LSTM + Attention (más elaborado) + Entrenamiento + Evaluación
# Pégala y ejecútala tal cual. Usa variables ya definidas en tu notebook.
# %%
# Imports (repetir import no rompe nada si ya están)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import numpy as np
import time, os
from collections import Counter
try:
    from tqdm.auto import tqdm
except:
    tqdm = lambda x: x

# device y logging mínimo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# parámetros adaptativos (útiles en CPU)
IS_CPU = device.type == "cpu"
BATCH_SIZE = 32 if IS_CPU else 64
EMBED_DIM = 256
HIDDEN_SIZE = 320
NUM_LAYERS = 2
DROPOUT = 0.35
BIDIRECTIONAL = True
EPOCHS = 23
PATIENCE = 4
GRAD_CLIP = 5.0

# util: pad index y número de clases / vocab
pad_idx = vocab.get('<pad>', 0)
NUM_CLASSES = len(label_encoder.classes_)
print(f"Vocab size: {vocab_size}  Pad idx: {pad_idx}  Num classes: {NUM_CLASSES}")

# --- collate: soporta (seq, label) o (seq, label, aux_dict) ---
def collate_batch(batch):
    # batch elements can be:
    #  (seq_tensor, label)
    #  (seq_tensor, label, aux)  where aux is dict of numpy/scalar features
    seqs = torch.stack([item[0] for item in batch])    # (B, L)
    labels = torch.tensor([int(item[1]) for item in batch], dtype=torch.long)
    lengths = (seqs != pad_idx).sum(dim=1)

    # handle optional aux
    aux_list = None
    if len(batch[0]) >= 3:
        # assume aux is dict; build matrix of floats
        try:
            aux_keys = list(batch[0][2].keys())
            aux_arr = np.stack([np.array([float(x[k]) for k in aux_keys]) for _, _, x in batch])
            aux_list = torch.tensor(aux_arr, dtype=torch.float)
        except Exception as e:
            aux_list = None

    # move to device
    seqs = seqs.to(device)
    lengths = lengths.to(device)
    labels = labels.to(device)
    if aux_list is not None:
        aux_list = aux_list.to(device)
        return seqs, lengths, labels, aux_list
    else:
        return seqs, lengths, labels

# DataLoaders (usa sampler si está definido)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=(sampler if 'sampler' in globals() and sampler is not None else None),
                          shuffle=(False if 'sampler' in globals() and sampler is not None else True),
                          collate_fn=collate_batch, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_batch)
print("Loaders: ", len(train_loader), len(val_loader), len(test_loader))

# --- calcular class weights a partir de train_dataset (si es posible) ---
def compute_class_weights(dataset):
    counts = Counter()
    for item in dataset:
        lab = int(item[1])
        counts[lab] += 1
    freqs = [counts[i] for i in range(NUM_CLASSES)]
    total = sum(freqs)
    # weight = total / (num_classes * freq)  (inversa proporcional)
    weights = [total / (NUM_CLASSES * (f if f>0 else 1)) for f in freqs]
    return torch.tensor(weights, dtype=torch.float).to(device)

try:
    class_weights = compute_class_weights(train_dataset)
    print("Class weights:", class_weights.cpu().numpy())
    criterion = nn.CrossEntropyLoss(weight=class_weights)
except Exception as e:
    print("No se pudieron calcular pesos de clase, uso CrossEntropyLoss simple.", str(e))
    criterion = nn.CrossEntropyLoss()

# --- Atención de pooling sobre salidas LSTM ---
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, attn_dim=128):
        super().__init__()
        self.linear = nn.Linear(hidden_dim, attn_dim, bias=True)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, h, mask):
        # h: (B, L, H)
        # mask: (B, L)  True for valid tokens
        # energy: (B, L, attn_dim)
        energy = torch.tanh(self.linear(h))          # (B,L,A)
        scores = self.v(energy).squeeze(-1)         # (B, L)
        scores = scores.masked_fill(~mask, float('-1e9'))
        alpha = torch.softmax(scores, dim=1)        # (B,L)
        context = torch.bmm(alpha.unsqueeze(1), h).squeeze(1)  # (B, H)
        return context, alpha

# --- Modelo: Embedding -> BiLSTM -> AttentionPooling -> FC (+ opcional aux features) ---
class LSTMWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, hidden_size=HIDDEN_SIZE,
                 num_layers=NUM_LAYERS, num_classes=NUM_CLASSES, dropout=DROPOUT,
                 bidirectional=BIDIRECTIONAL, padding_idx=0, use_aux=False, aux_dim=0, attn_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        # optional: try to init embeddings if a global `pretrained_embeddings` exists
        try:
            if 'pretrained_embeddings' in globals():
                with torch.no_grad():
                    emb_tensor = torch.tensor(pretrained_embeddings, dtype=torch.float)
                    if emb_tensor.shape[0] == vocab_size and emb_tensor.shape[1] == embed_dim:
                        self.embedding.weight.data.copy_(emb_tensor)
                        print("Loaded pretrained embeddings into embedding layer.")
        except Exception:
            pass

        self.dropout_in = nn.Dropout(dropout*0.5)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers, batch_first=True,
                            bidirectional=bidirectional, dropout=dropout if num_layers>1 else 0.0)
        factor = 2 if bidirectional else 1
        self.attention = AttentionPooling(hidden_dim=hidden_size*factor, attn_dim=attn_dim)
        self.use_aux = use_aux
        fc_input = hidden_size*factor + (aux_dim if use_aux else 0)
        self.fc = nn.Sequential(
            nn.Linear(fc_input, fc_input//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_input//2, num_classes)
        )

    def forward(self, x, lengths, aux=None):
        # x: (B, L)
        mask = (x != pad_idx)  # (B, L) boolean
        emb = self.embedding(x)        # (B, L, E)
        emb = self.dropout_in(emb)

        # pack
        lengths_cpu = lengths.cpu()
        packed = pack_padded_sequence(emb, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out_unpacked, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))  # (B,L,H)

        # Attention pooling
        context, attn_weights = self.attention(out_unpacked, mask)  # (B, H)

        if self.use_aux and aux is not None:
            context = torch.cat([context, aux], dim=1)

        logits = self.fc(context)  # (B, num_classes)
        return logits, attn_weights

# --- instancia del modelo (si tus datasets traen aux features, usamos aux_dim)
aux_dim = 0
# detect if dataloader returns aux (by peeking one batch)
it = iter(train_loader)
sample = next(it)
# sample can be (seqs,lengths,labels) or (seqs,lengths,labels,aux)
has_aux = (len(sample) == 4)
if has_aux:
    aux_dim = sample[3].shape[1]
print("Aux features present:", has_aux, " aux_dim=", aux_dim)
# create model
model = LSTMWithAttention(vocab_size=vocab_size, embed_dim=EMBED_DIM, hidden_size=HIDDEN_SIZE,
                          num_layers=NUM_LAYERS, num_classes=NUM_CLASSES, dropout=DROPOUT,
                          bidirectional=BIDIRECTIONAL, padding_idx=pad_idx, use_aux=has_aux, aux_dim=aux_dim).to(device)
print(model)

# --- optimizer, scheduler, etc. ---
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
# gradient accumulation if CPU or low VRAM (helps emulate larger batches)
GRAD_ACCUM = 1 if not IS_CPU else 2

# --- util evaluación ---
def evaluate_full(model, dataloader):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            if has_aux:
                seqs, lengths, labels, aux = batch
                logits, attn = model(seqs, lengths, aux)
            else:
                seqs, lengths, labels = batch
                logits, attn = model(seqs, lengths, None)
            loss = criterion(logits, labels)
            losses.append(loss.item() * seqs.size(0))
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    avg_loss = sum(losses) / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, macro_f1, all_labels, all_preds

# --- entrenamiento con early stopping y checkpoint ---
best_val_loss = float("inf")
no_improve = 0
best_path = "./lstm_attn_best.pt"
os.makedirs(os.path.dirname(best_path) or ".", exist_ok=True)
start_time = time.time()

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    nbatch = 0
    optimizer.zero_grad()
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for i, batch in enumerate(loop):
        if has_aux:
            seqs, lengths, labels, aux = batch
            logits, _ = model(seqs, lengths, aux)
        else:
            seqs, lengths, labels = batch
            logits, _ = model(seqs, lengths, None)

        loss = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM
        nbatch += 1

        if (i+1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad()

    train_loss = running_loss / max(1, nbatch)
    val_loss, val_acc, val_f1, _, _ = evaluate_full(model, val_loader)
    scheduler.step(val_loss)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

    # checkpoint
    if val_loss + 1e-8 < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save({"model_state": model.state_dict(),
                    "vocab": vocab,
                    "label_classes": list(label_encoder.classes_),
                    "epoch": epoch}, best_path)
        print("  -> Mejor modelo guardado.")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping tras {PATIENCE} épocas sin mejora.")
            break

total_min = (time.time() - start_time) / 60.0
print(f"Entrenamiento completado en {total_min:.2f} minutos.")

# --- evaluación final con el mejor checkpoint ---
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    test_loss, test_acc, test_f1, y_true, y_pred = evaluate_full(model, test_loader)
    print("\n== RESULTADOS FINALES (test) ==")
    print(f"Test loss: {test_loss:.4f}  Test acc: {test_acc:.4f}  Test macro-F1: {test_f1:.4f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
else:
    print("No checkpoint encontrado para evaluación final.")


C:\Users\irvin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Vocab size: 11495  Pad idx: 0  Num classes: 3
Loaders:  805 87 87
Class weights: [0.584196  1.314083  1.8966025]
Aux features present: False  aux_dim= 0
LSTMWithAttention(
  (embedding): Embedding(11495, 256, padding_idx=0)
  (dropout_in): Dropout(p=0.175, inplace=False)
  (lstm): LSTM(256, 320, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (attention): AttentionPooling(
    (linear): Linear(in_features=640, out_features=128, bias=True)
    (v): Linear(in_features=128, out_features=1, bias=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=640, out_features=320, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.35, inplace=False)
    (3): Linear(in_features=320, out_features=3, bias=True)
  )
)


Epoch 1/23: 100%|██████████| 805/805 [16:23<00:00,  1.22s/it]


Epoch 01 | train_loss=0.7998 val_loss=1.0112 val_acc=0.3579 val_f1=0.3486
  -> Mejor modelo guardado.


Epoch 2/23: 100%|██████████| 805/805 [13:01<00:00,  1.03it/s]


...

Epoch 03 | train_loss=0.5501 val_loss=1.1218 val_acc=0.3985 val_f1=0.3969


Epoch 4/23: 100%|██████████| 805/805 [13:32<00:00,  1.01s/it]


Epoch 04 | train_loss=0.4541 val_loss=1.2071 val_acc=0.4399 val_f1=0.4433


Epoch 5/23: 100%|██████████| 805/805 [13:23<00:00,  1.00it/s]


Epoch 05 | train_loss=0.3514 val_loss=1.3420 val_acc=0.5044 val_f1=0.4950
Early stopping tras 4 épocas sin mejora.
Entrenamiento completado en 71.88 minutos.

== RESULTADOS FINALES (test) ==
Test loss: 1.0199  Test acc: 0.3657  Test macro-F1: 0.3568

Classification report:
              precision    recall  f1-score   support

    Adequate       0.81      0.06      0.11      3147
   Effective       0.45      0.76      0.57      1399
 Ineffective       0.26      0.78      0.39       969

    accuracy                           0.37      5515
   macro avg       0.51      0.54      0.36      5515
weighted avg       0.62      0.37      0.28      5515

Confusion matrix:
[[ 190 1111 1846]
 [  19 1070  310]
 [  25  187  757]]


In [7]:
# %% [markdown]
# CELDA: reentrenamiento con WeightedRandomSampler + FocalLoss + checkpoint por macro-F1
# Pégala y ejecútala tal cual. Usa las variables ya definidas en tu notebook.
# %%
import torch, time, os, numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from collections import Counter
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
try:
    from tqdm.auto import tqdm
except:
    tqdm = lambda x: x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------- 1) sanity checks ----------
print("Label classes:", list(label_encoder.classes_))
# count labels in train
train_counts = Counter(int(item[1]) for item in train_dataset)
print("Train label counts:", train_counts)

# ---------- 2) build WeightedRandomSampler to balance classes ----------
num_samples = len(train_dataset)
# weight per class = 1 / freq
class_weights = {cls: 1.0 / max(1, train_counts.get(cls, 0)) for cls in range(len(label_encoder.classes_))}
sample_weights = [class_weights[int(item[1])] for item in train_dataset]
sample_weights = torch.DoubleTensor(sample_weights)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=num_samples, replacement=True)
print("Sampler built. Example class weight:", {k: float(class_weights[k]) for k in class_weights})

# ---------- 3) collate (compatible con aux opcional) ----------
pad_idx = vocab.get('<pad>', 0)
def collate_batch(batch):
    seqs = torch.stack([item[0] for item in batch])
    labels = torch.tensor([int(item[1]) for item in batch], dtype=torch.long)
    lengths = (seqs != pad_idx).sum(dim=1)
    if len(batch[0]) >= 3:
        try:
            aux_keys = list(batch[0][2].keys())
            aux_arr = np.stack([np.array([float(x[k]) for k in aux_keys]) for _, _, x in batch])
            aux_list = torch.tensor(aux_arr, dtype=torch.float)
            return seqs.to(device), lengths.to(device), labels.to(device), aux_list.to(device)
        except:
            pass
    return seqs.to(device), lengths.to(device), labels.to(device)

# loaders (train usa sampler balanceado)
BATCH_SIZE = 32 if device.type == 'cpu' else 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_batch, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_batch)
print("Loaders len:", len(train_loader), len(val_loader), len(test_loader))

# ---------- 4) Focal Loss implementation ----------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean', eps=1e-8):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            alpha = torch.tensor(alpha, dtype=torch.float)
        self.alpha = alpha
        self.reduction = reduction
        self.eps = eps

    def forward(self, inputs, targets):
        # inputs: logits (B, C), targets: (B,)
        probs = torch.softmax(inputs, dim=1)
        probs = probs.clamp(self.eps, 1.0 - self.eps)
        pt = probs[range(len(targets)), targets]  # (B,)
        log_pt = torch.log(pt)
        loss = - ((1 - pt) ** self.gamma) * log_pt
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                alpha = self.alpha.to(inputs.device)
            else:
                alpha = self.alpha
            at = alpha[targets]
            loss = at * loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

# compute alpha from class frequencies (smoothed)
freqs = np.array([train_counts.get(i,0) for i in range(len(label_encoder.classes_))], dtype=np.float32)
alpha = (freqs.sum() / (len(freqs) * (freqs + 1e-8)))  # inverse-frequency smoothed
alpha = alpha / alpha.sum()  # normalize
print("Focal alpha:", alpha)

criterion = FocalLoss(gamma=2.0, alpha=alpha)

# ---------- 5) rebuild model (same architecture LSTM+Attention used antes) ----------
# We'll reuse the same class definitions you used antes (LSTMWithAttention). Si no están en memoria, definimos una versión rápida.

# (si tienes definida LSTMWithAttention previamente en el notebook, el siguiente bloque la sobrescribe con la misma arquitectura)
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, attn_dim=128):
        super().__init__()
        self.linear = nn.Linear(hidden_dim, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, h, mask):
        energy = torch.tanh(self.linear(h))
        scores = self.v(energy).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-1e9'))
        alpha = torch.softmax(scores, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), h).squeeze(1)
        return context, alpha

class LSTMWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_size=320, num_layers=2,
                 num_classes=3, dropout=0.35, bidirectional=True, padding_idx=0, use_aux=False, aux_dim=0, attn_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers, batch_first=True,
                            bidirectional=bidirectional, dropout=dropout if num_layers>1 else 0.0)
        factor = 2 if bidirectional else 1
        self.attention = AttentionPooling(hidden_dim=hidden_size*factor, attn_dim=attn_dim)
        self.use_aux = use_aux
        fc_input = hidden_size*factor + (aux_dim if use_aux else 0)
        self.fc = nn.Sequential(
            nn.Linear(fc_input, fc_input//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_input//2, num_classes)
        )

    def forward(self, x, lengths, aux=None):
        mask = (x != pad_idx)
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out_unpacked, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        context, attn = self.attention(out_unpacked, mask)
        if self.use_aux and aux is not None:
            context = torch.cat([context, aux], dim=1)
        logits = self.fc(context)
        return logits, attn

# detect aux in train_dataset
sample = train_dataset[0]
has_aux = len(sample) >= 3 and isinstance(sample[2], dict)
aux_dim = 0
if has_aux:
    aux_dim = len(list(sample[2].keys()))
print("Has aux?", has_aux, " aux_dim=", aux_dim)

model = LSTMWithAttention(vocab_size=vocab_size, embed_dim=256, hidden_size=320, num_layers=2,
                          num_classes=len(label_encoder.classes_), dropout=0.35, bidirectional=True,
                          padding_idx=pad_idx, use_aux=has_aux, aux_dim=aux_dim).to(device)

# optimizer + scheduler
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)  # note: monitor macro-F1

# ---------- 6) entrenamiento: monitor macro-F1, checkpoint por mejor macro-F1 ----------
best_val_f1 = -1.0
no_improve = 0
PATIENCE = 4
best_path = "./lstm_balanced_focal_best.pt"
os.makedirs(os.path.dirname(best_path) or ".", exist_ok=True)

def evaluate_return_stats(model, dataloader):
    model.eval()
    losses = []
    preds = []
    trues = []
    with torch.no_grad():
        for batch in dataloader:
            if len(batch) == 4:
                seqs, lengths, labels, aux = batch
                logits, _ = model(seqs, lengths, aux)
            else:
                seqs, lengths, labels = batch
                logits, _ = model(seqs, lengths, None)
            loss = criterion(logits, labels)
            losses.append(loss.item() * seqs.size(0))
            p = torch.argmax(logits, dim=1).cpu().numpy()
            preds.extend(p.tolist())
            trues.extend(labels.cpu().numpy().tolist())
    avg_loss = sum(losses) / len(dataloader.dataset)
    acc = accuracy_score(trues, preds)
    macro = f1_score(trues, preds, average='macro')
    return avg_loss, acc, macro, trues, preds

start_time = time.time()
for epoch in range(1, 15):
    model.train()
    running_loss = 0.0
    nb = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}")
    for batch in loop:
        if len(batch) == 4:
            seqs, lengths, labels, aux = batch
            logits, _ = model(seqs, lengths, aux)
        else:
            seqs, lengths, labels = batch
            logits, _ = model(seqs, lengths, None)

        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        nb += 1

    train_loss = running_loss / max(1, nb)
    val_loss, val_acc, val_f1, y_true, y_pred = evaluate_return_stats(model, val_loader)
    # scheduler monitors macro-F1 (mode='max' above), so step with val_f1
    scheduler.step(val_f1)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macroF1={val_f1:.4f}")

    # checkpoint on macro-F1
    if val_f1 > best_val_f1 + 1e-5:
        best_val_f1 = val_f1
        no_improve = 0
        torch.save({"model_state": model.state_dict(), "epoch": epoch, "vocab": vocab, "label_classes": list(label_encoder.classes_)}, best_path)
        print("  -> Nuevo mejor macro-F1, checkpoint guardado.")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping tras {PATIENCE} épocas sin mejora (macro-F1).")
            break

total_min = (time.time() - start_time) / 60.0
print(f"Entrenamiento finalizado en {total_min:.2f} minutos.")

# ---------- 7) evaluación final ----------
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    test_loss, test_acc, test_f1, y_true, y_pred = evaluate_return_stats(model, test_loader)
    print("\n== RESULTADO FINAL (test) ==")
    print(f"Test loss: {test_loss:.4f}  Test acc: {test_acc:.4f}  Test macro-F1: {test_f1:.4f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
else:
    print("No se encontró checkpoint guardado para evaluación final.")


Device: cpu
Label classes: ['Adequate', 'Effective', 'Ineffective']
Train label counts: Counter({0: 14684, 1: 6528, 2: 4523})
Sampler built. Example class weight: {0: 6.810133478616181e-05, 1: 0.00015318627450980392, 2: 0.00022109219544550078}
Loaders len: 805 87 87
Focal alpha: [0.15394315 0.34627774 0.49977913]
Has aux? False  aux_dim= 0


Epoch 1: 100%|██████████| 805/805 [12:45<00:00,  1.05it/s]


Epoch 01 | train_loss=0.1087 val_loss=0.1160 val_acc=0.3414 val_macroF1=0.3208
  -> Nuevo mejor macro-F1, checkpoint guardado.


Epoch 2: 100%|██████████| 805/805 [13:38<00:00,  1.02s/it]


Epoch 02 | train_loss=0.0855 val_loss=0.1179 val_acc=0.3835 val_macroF1=0.3879
  -> Nuevo mejor macro-F1, checkpoint guardado.


...

Epoch 12 | train_loss=0.0115 val_loss=0.3188 val_acc=0.5327 val_macroF1=0.5050


Epoch 13: 100%|██████████| 805/805 [12:23<00:00,  1.08it/s]


Epoch 13 | train_loss=0.0097 val_loss=0.3611 val_acc=0.5558 val_macroF1=0.5154


Epoch 14: 100%|██████████| 805/805 [12:29<00:00,  1.07it/s]


Epoch 14 | train_loss=0.0062 val_loss=0.3876 val_acc=0.5688 val_macroF1=0.5185
Early stopping tras 4 épocas sin mejora (macro-F1).
Entrenamiento finalizado en 180.98 minutos.

== RESULTADO FINAL (test) ==
Test loss: 0.3175  Test acc: 0.5344  Test macro-F1: 0.5047

Classification report:
              precision    recall  f1-score   support

    Adequate       0.66      0.52      0.58      3147
   Effective       0.51      0.64      0.56      1399
 Ineffective       0.32      0.42      0.36       969

    accuracy                           0.53      5515
   macro avg       0.50      0.53      0.50      5515
weighted avg       0.56      0.53      0.54      5515

Confusion matrix:
[[1644  766  737]
 [ 383  896  120]
 [ 451  111  407]]
